## Training Environment:
All model training was conducted using Google Colab with GPU acceleration (NVIDIA L4 GPU).

- System RAM: 53.0 GB
- GPU RAM: 22.5 GB

In [ ]:
# ===== Cell 1: Upload ZIP (optional) =====
# If you already see PlantVillage.zip in the left Files panel, skip this cell.
from google.colab import files
uploaded = files.upload()  # choose PlantVillage.zip

In [ ]:
# ===== Cell 2: Unzip PlantVillage.zip safely (AUTO detect nesting) =====
import os, zipfile, shutil

ZIP_PATH = "/content/PlantVillage.zip"   # must match the file name in Colab
OUT_ROOT = "/content/PlantVillage"      # where we extract

# Clean old extract (optional but recommended)
if os.path.exists(OUT_ROOT):
    shutil.rmtree(OUT_ROOT)
os.makedirs(OUT_ROOT, exist_ok=True)

# Check zip is valid
assert os.path.exists(ZIP_PATH), f"ZIP not found at: {ZIP_PATH}"
assert zipfile.is_zipfile(ZIP_PATH), "This file is not a valid ZIP (maybe corrupted or wrong file type)."

# Extract
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(OUT_ROOT)

print("✅ Extracted to:", OUT_ROOT)
print("Top level:", os.listdir(OUT_ROOT)[:20])

In [ ]:
# ===== Cell 3: Find the real BASE_DIR that contains train/ and val/ =====
import os

def find_base_with_train_val(root="/content/PlantVillage"):
    # Look for a folder that directly contains both 'train' and 'val'
    for dirpath, dirnames, filenames in os.walk(root):
        dset = set(dirnames)
        if "train" in dset and "val" in dset:
            return dirpath
    return None

BASE_DIR = find_base_with_train_val("/content/PlantVillage")
print("BASE_DIR found:", BASE_DIR)

assert BASE_DIR is not None, "❌ Could not find a folder containing BOTH train/ and val/. Your ZIP structure is different."

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")

print("TRAIN_DIR:", TRAIN_DIR, "exists?", os.path.isdir(TRAIN_DIR))
print("VAL_DIR  :", VAL_DIR, "exists?", os.path.isdir(VAL_DIR))
print("\nTrain folders sample:", os.listdir(TRAIN_DIR)[:10])
print("Val folders sample  :", os.listdir(VAL_DIR)[:10])

In [ ]:
# ===== Cell 4: Imports + GPU + seeds (works with your existing TRAIN_DIR / VAL_DIR) =====
import os, json, math, random, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_auc_score, roc_curve, auc
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if gpus:
    try:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
        print("Memory growth enabled.")
    except Exception as e:
        print("Could not set memory growth:", e)

AUTOTUNE = tf.data.AUTOTUNE

print("TRAIN_DIR =", TRAIN_DIR)
print("VAL_DIR   =", VAL_DIR)

In [ ]:
# ===== Cell 5: Build file lists from TRAIN_DIR + VAL_DIR, and class_names from TRAIN only =====
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

train_root = Path(TRAIN_DIR)
val_root   = Path(VAL_DIR)

assert train_root.exists() and train_root.is_dir(), "TRAIN_DIR is not valid."
assert val_root.exists() and val_root.is_dir(), "VAL_DIR is not valid."

class_dirs = sorted([p for p in train_root.iterdir() if p.is_dir()])
assert len(class_dirs) >= 2, "Need at least 2 class folders inside TRAIN_DIR."

class_names = [p.name for p in class_dirs]
label_to_idx = {c:i for i,c in enumerate(class_names)}
NUM_CLASSES = len(class_names)

def collect_files(root: Path):
    rows = []
    for c in sorted([p for p in root.iterdir() if p.is_dir()]):
        if c.name not in label_to_idx:
            continue
        for fp in c.rglob("*"):
            if fp.is_file() and fp.suffix.lower() in IMG_EXTS:
                rows.append((str(fp), c.name))
    return pd.DataFrame(rows, columns=["filepath", "label"])

train_df_full = collect_files(train_root).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df_full   = collect_files(val_root).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Classes:", NUM_CLASSES)
print("Train images:", len(train_df_full))
print("Val images:", len(val_df_full))
print(train_df_full["label"].value_counts().head())

In [ ]:
# ===== Cell 6: Split provided VAL into (val + test) so you get a true test set =====
# If you want a different ratio, change TEST_FROM_VAL.
TEST_FROM_VAL = 0.50  # 50% of provided val becomes test

val_df, test_df = train_test_split(
    val_df_full,
    test_size=TEST_FROM_VAL,
    random_state=SEED,
    stratify=val_df_full["label"]
)

train_df = train_df_full.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Final splits:")
print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))

In [ ]:
# ===== Cell 7: Leakage check (filename overlap + optional MD5) + AUTO-FIX =====

from pathlib import Path
import hashlib
import pandas as pd

RUN_MD5_CHECK = True   # set False once you're done (MD5 can be slow on big datasets)

def _basename_set(df: pd.DataFrame) -> set:
    return set(Path(p).name for p in df["filepath"].values)

train_names = _basename_set(train_df)
val_names   = _basename_set(val_df)
test_names  = _basename_set(test_df)

print("Name overlaps (fast check):")
print("train/val :", len(train_names & val_names))
print("train/test:", len(train_names & test_names))
print("val/test  :", len(val_names & test_names))

if RUN_MD5_CHECK:
    def md5_of_file(path: str, chunk_size: int = 1 << 20) -> str:
        h = hashlib.md5()
        with open(path, "rb") as f:
            while True:
                b = f.read(chunk_size)
                if not b:
                    break
                h.update(b)
        return h.hexdigest()

    # Build one MD5 per unique filepath (faster than recomputing repeatedly)
    all_paths = pd.concat([train_df["filepath"], val_df["filepath"], test_df["filepath"]]).astype(str).unique()
    fp2md5 = {}
    for p in all_paths:
        fp2md5[p] = md5_of_file(p)

    def add_hashes(df: pd.DataFrame, split: str) -> pd.DataFrame:
        out = df.copy()
        out["md5"] = out["filepath"].astype(str).map(fp2md5)
        out["split"] = split
        return out

    all_h = pd.concat(
        [add_hashes(train_df, "train"), add_hashes(val_df, "val"), add_hashes(test_df, "test")],
        ignore_index=True
    )

    leaks = all_h.groupby("md5")["split"].nunique()
    leak_md5 = leaks[leaks > 1].index

    print("\nMD5 leaks across splits:", int(leak_md5.shape[0]))

    if len(leak_md5) > 0:
        # Show a sample so you can verify what's going on
        display(all_h[all_h["md5"].isin(leak_md5)].sort_values(["md5", "split"]).head(25))

        # ---- AUTO-FIX RULES ----
        # 1) If a duplicate exists in train and (val/test) => DROP from val/test.
        # 2) If a duplicate exists in val and test (but not train) => DROP from test.
        train_md5 = set(all_h.loc[all_h["split"] == "train", "md5"].unique())

        # Drop from val/test anything that is also in train
        val_df  = val_df[~val_df["filepath"].astype(str).map(fp2md5).isin(train_md5)].reset_index(drop=True)
        test_df = test_df[~test_df["filepath"].astype(str).map(fp2md5).isin(train_md5)].reset_index(drop=True)

        # Now handle val-test duplicates (keep val, drop from test)
        val_md5  = set(val_df["filepath"].astype(str).map(fp2md5).unique())
        test_md5 = set(test_df["filepath"].astype(str).map(fp2md5).unique())
        val_test_dupes = val_md5 & test_md5

        if len(val_test_dupes) > 0:
            test_df = test_df[~test_df["filepath"].astype(str).map(fp2md5).isin(val_test_dupes)].reset_index(drop=True)

        # Re-check after fix (no recompute, re-use fp2md5)
        all_h2 = pd.concat(
            [add_hashes(train_df, "train"), add_hashes(val_df, "val"), add_hashes(test_df, "test")],
            ignore_index=True
        )
        leaks2 = all_h2.groupby("md5")["split"].nunique()
        leak_md52 = leaks2[leaks2 > 1].index

        print("\nAfter AUTO FIX:")
        print("MD5 leaks across splits (after fix):", int(leak_md52.shape[0]))
        print(f"New sizes -> train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")
else:
    print("\nRUN_MD5_CHECK=False: skipping MD5 leakage check.")

In [ ]:
# ===== Cell 8: EDA plot =====
def plot_split_counts(train_df, val_df, test_df):
    plt.figure(figsize=(10,4))
    for d, name in [(train_df,"train"), (val_df,"val"), (test_df,"test")]:
        counts = d["label"].value_counts().reindex(class_names, fill_value=0)
        plt.plot(range(NUM_CLASSES), counts.values, marker="o", linewidth=1, label=name)
    plt.xticks(range(NUM_CLASSES), class_names, rotation=90)
    plt.ylabel("count")
    plt.title("Class distribution per split")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_split_counts(train_df, val_df, test_df)

In [ ]:
# ===== Cell 9: Class weights from TRAIN =====
y_train_idx = train_df["label"].map(label_to_idx).values
cw = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=y_train_idx
)
class_weights = {i: float(w) for i, w in enumerate(cw)}
print("class_weights ready ✅")

In [ ]:
# ===== Cell 10: tf.data pipeline =====
IMG_SIZE = (224, 224)
BATCH_SIZE = 64

def load_image(path, label_idx):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    y = tf.one_hot(label_idx, NUM_CLASSES)
    return img, y

def make_ds(d: pd.DataFrame, training: bool):
    paths = d["filepath"].values
    labels = d["label"].map(label_to_idx).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(d), 8000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df, training=False)
test_ds  = make_ds(test_df, training=False)

print("Batches:",
      tf.data.experimental.cardinality(train_ds).numpy(),
      tf.data.experimental.cardinality(val_ds).numpy(),
      tf.data.experimental.cardinality(test_ds).numpy())

In [ ]:
# ===== Cell 11: Visualize a batch =====
def show_batch(ds, n=9):
    plt.figure(figsize=(8,8))
    for images, labels in ds.take(1):
        for i in range(min(n, images.shape[0])):
            ax = plt.subplot(int(math.sqrt(n)), int(math.sqrt(n)), i+1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(class_names[int(tf.argmax(labels[i]))])
            plt.axis("off")
    plt.tight_layout()
    plt.show()

show_batch(train_ds, n=9)

In [ ]:
# ===== Cell 12: Augmentation + normalization =====
data_augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.12),
], name="augment")

normalizer = layers.Rescaling(1./255, name="rescale")

In [ ]:
# ===== Cell 13: Helpers (curves + confusion matrix + ROC/AUC) =====
def plot_history(hist, title="Training curves"):
    h = hist.history

    plt.figure(figsize=(7,4))
    plt.plot(h.get("accuracy", []), label="train_acc")
    plt.plot(h.get("val_accuracy", []), label="val_acc")
    plt.title(title)
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7,4))
    plt.plot(h.get("loss", []), label="train_loss")
    plt.plot(h.get("val_loss", []), label="val_loss")
    plt.title(title)
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_confusion(cm, title):
    plt.figure(figsize=(7,6))
    plt.imshow(cm)
    plt.title(title)
    plt.xticks(range(NUM_CLASSES), class_names, rotation=90)
    plt.yticks(range(NUM_CLASSES), class_names)
    plt.xlabel("Pred")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

def plot_multiclass_roc(y_true_idx, probs, title="ROC"):
    # For many classes (PlantVillage can be ~38), plot micro + macro + top-5 most common classes
    y_true_oh = keras.utils.to_categorical(y_true_idx, NUM_CLASSES)

    fpr = {}
    tpr = {}
    roc_auc = {}

    for i in range(NUM_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_true_oh[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # micro-average
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true_oh.ravel(), probs.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    # macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= NUM_CLASSES
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    plt.figure(figsize=(7,5))
    plt.plot(fpr["micro"], tpr["micro"], label=f"micro (AUC={roc_auc['micro']:.3f})")
    plt.plot(fpr["macro"], tpr["macro"], label=f"macro (AUC={roc_auc['macro']:.3f})")

    # top-5 classes by support
    counts = pd.Series(y_true_idx).value_counts()
    topk = list(counts.index[:5])
    for i in topk:
        plt.plot(fpr[i], tpr[i], linewidth=1, label=f"{class_names[i]} (AUC={roc_auc[i]:.3f})")

    plt.title(title)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

def eval_model(model, ds, y_true_idx, title="Model"):
    probs = model.predict(ds, verbose=0)
    y_pred = probs.argmax(axis=1)

    acc = accuracy_score(y_true_idx, y_pred)
    f1m = f1_score(y_true_idx, y_pred, average="macro")
    f1w = f1_score(y_true_idx, y_pred, average="weighted")

    print(f"\n== {title} ==")
    print("Accuracy:", acc)
    print("F1 macro:", f1m)
    print("F1 weighted:", f1w)
    print("\nClassification report:")
    print(classification_report(y_true_idx, y_pred, target_names=class_names, digits=4))

    cm = confusion_matrix(y_true_idx, y_pred)
    plot_confusion(cm, f"Confusion Matrix — {title}")

    # AUC macro OVR
    try:
        y_true_oh = keras.utils.to_categorical(y_true_idx, NUM_CLASSES)
        auc_macro = roc_auc_score(y_true_oh, probs, multi_class="ovr", average="macro")
    except Exception:
        auc_macro = None

    # ROC plot
    try:
        plot_multiclass_roc(y_true_idx, probs, title=f"ROC — {title}")
    except Exception as e:
        print("ROC plot skipped:", e)

    return {"acc": acc, "f1_macro": f1m, "f1_weighted": f1w, "auc_macro_ovr": auc_macro}

In [ ]:
# ===== Cell 14: CNN =====
def build_scratch_cnn():
    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = data_augment(inputs)
    x = normalizer(x)

    # --- Feature extraction blocks ---
    # Block 1: 32 filters — learn low-level features (edges, colours)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)   # stabilise activations after each block
    x = layers.MaxPooling2D()(x)

    # Block 2: 64 filters — learn mid-level textures
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    # Block 3: 128 filters — learn high-level patterns (leaf spots, lesions)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Dropout(0.30)(x)
    x = layers.GlobalAveragePooling2D()(x)  # compress spatial maps to 1-D feature vector

    # --- Fully-connected classifier head (perceptron layers) ---
    # Dense(512): first perceptron layer — wide representation to capture all 38 class patterns
    x = layers.Dense(512, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)

    # Dense(256): second perceptron layer — compress to more abstract representation
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.35)(x)

    # Output layer: 38 nodes (one perceptron per disease class), softmax for probabilities
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="ScratchCNN")
    model.compile(
        optimizer=keras.optimizers.Adam(3e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

scratch = build_scratch_cnn()
scratch.summary()


In [ ]:
# ===== Cell 15: CNN Hyperparameter Tuning =====
import gc

TUNE_SAMPLES  = 30000
TUNE_VAL_SAMP = 5000
TUNE_EPOCHS   = 8
TUNE_BATCH_SIZE = 256  # bigger = faster + more GPU RAM

tune_tr_df  = train_df.sample(n=TUNE_SAMPLES,  random_state=SEED).reset_index(drop=True)
tune_val_df = val_df.sample(n=TUNE_VAL_SAMP,   random_state=SEED+1).reset_index(drop=True)

def make_tune_ds(d, training=False):
    paths  = d["filepath"].values
    labels = d["label"].map(label_to_idx).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(d), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(TUNE_BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

tune_tr_ds  = make_tune_ds(tune_tr_df,  training=True)
tune_val_ds = make_tune_ds(tune_val_df, training=False)

def build_cnn_tune(dense1, dense2, dropout1, dropout2, lr):
    inp = keras.Input(shape=(*IMG_SIZE, 3))
    x   = data_augment(inp)
    x   = normalizer(x)

    x = layers.Conv2D(32,  3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64,  3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Dropout(0.30)(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(dense1, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout1)(x)

    x = layers.Dense(dense2, activation="relu")(x)
    x = layers.Dropout(dropout2)(x)

    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    m = keras.Model(inp, out)
    m.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return m

configs = [
    {"dense1": 512, "dense2": 256, "dropout1": 0.40, "dropout2": 0.35, "lr": 1e-3},
    {"dense1": 512, "dense2": 256, "dropout1": 0.35, "dropout2": 0.30, "lr": 5e-4},
    {"dense1": 384, "dense2": 192, "dropout1": 0.40, "dropout2": 0.35, "lr": 1e-3},
    {"dense1": 384, "dense2": 192, "dropout1": 0.35, "dropout2": 0.30, "lr": 5e-4},
    {"dense1": 256, "dense2": 128, "dropout1": 0.40, "dropout2": 0.35, "lr": 1e-3},
    {"dense1": 256, "dense2": 128, "dropout1": 0.35, "dropout2": 0.30, "lr": 3e-4},
]

rows = []
for k, cfg in enumerate(configs, 1):
    tf.random.set_seed(SEED)

    m = build_cnn_tune(**cfg)
    h = m.fit(
        tune_tr_ds,
        validation_data=tune_val_ds,
        epochs=TUNE_EPOCHS,
        class_weight=class_weights,
        verbose=0
    )
    best_acc  = float(max(h.history["val_accuracy"]))
    best_loss = float(min(h.history["val_loss"]))
    rows.append({**cfg, "val_accuracy": round(best_acc,4), "val_loss": round(best_loss,4)})
    print(f"[{k}/{len(configs)}] dense1={cfg['dense1']} dense2={cfg['dense2']} "
          f"drop={cfg['dropout1']}/{cfg['dropout2']} lr={cfg['lr']} "
          f"-> val_acc={best_acc:.4f}")
    del m
    gc.collect()

tune_results = pd.DataFrame(rows).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
display(tune_results)

best = tune_results.iloc[0]
print(f"\n✅ Best config: dense1={int(best.dense1)}, dense2={int(best.dense2)}, "
      f"dropout1={best.dropout1}, dropout2={best.dropout2}, lr={best.lr}")



In [ ]:
# ===== Cell 16: Train CNN
EPOCHS_SCRATCH = 20

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    ),
]

hist_scratch = scratch.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_SCRATCH,
    class_weight=class_weights,
    callbacks=cb,
    verbose=1
)

plot_history(hist_scratch, "ScratchCNN (fixed)")

In [ ]:
# ===== Cell 17: Evaluate CNN
y_test_idx = test_df["label"].map(label_to_idx).values
metrics_scratch = eval_model(scratch, test_ds, y_test_idx, title="ScratchCNN")
print("Scratch metrics:", metrics_scratch)

In [ ]:
# ===== Cell 18: Hyperparameter tuning evidence


import time, gc

# Toggle this ON to actually run the tuning and generate the output table.
RUN_TUNING = True

# Defaults (matches your original EfficientNet settings)
EFFNET_HEAD_UNITS = 0
EFFNET_DROPOUT   = 0.25
EFFNET_LR_FROZEN = 1e-3
EFFNET_LR_FT     = 1e-4

if RUN_TUNING:
    # Keep it fast: tune on a subset of batches (still uses your real preprocessing pipeline)
    TUNE_TRAIN_BATCHES = 200
    TUNE_VAL_BATCHES   = 60
    TUNE_EPOCHS        = 2

    tune_train = train_ds.take(TUNE_TRAIN_BATCHES)
    tune_val   = val_ds.take(TUNE_VAL_BATCHES)

    def build_effnet_variant(head_units=0, dropout=0.25, lr=1e-3):
        base_tmp = keras.applications.EfficientNetB0(
            include_top=False,
            weights="imagenet",
            input_shape=(*IMG_SIZE, 3),
        )
        base_tmp.trainable = False

        inputs = keras.Input(shape=(*IMG_SIZE, 3))
        x = data_augment(inputs)

        # EfficientNet expects [0..255] float inputs; no extra Rescaling here.
        x = base_tmp(x, training=False)

        x = layers.GlobalAveragePooling2D()(x)
        if head_units and head_units > 0:
            x = layers.Dense(head_units, activation="relu")(x)

        x = layers.Dropout(dropout)(x)
        outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

        model = keras.Model(inputs, outputs)
        model.compile(
            optimizer=keras.optimizers.Adam(lr),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
        )
        return model

    configs = [
        {"tag":"baseline",     "head_units":0,   "dropout":0.25, "lr":1e-3},
        {"tag":"+dense128",    "head_units":128, "dropout":0.25, "lr":1e-3},
        {"tag":"+dense128_d35","head_units":128, "dropout":0.35, "lr":1e-3},
        {"tag":"+dense128_lr","head_units":128, "dropout":0.25, "lr":3e-4},
        {"tag":"dropout_0.35", "head_units":0,   "dropout":0.35, "lr":1e-3},
        {"tag":"lr_3e-4",      "head_units":0,   "dropout":0.25, "lr":3e-4},
    ]

    rows = []
    for cfg in configs:
        tf.keras.backend.clear_session()
        gc.collect()

        t0 = time.time()
        m = build_effnet_variant(cfg["head_units"], cfg["dropout"], cfg["lr"])

        h = m.fit(
            tune_train,
            validation_data=tune_val,
            epochs=TUNE_EPOCHS,
            verbose=0
        )

        best_val_acc = float(max(h.history.get("val_accuracy", [0.0])))
        best_val_loss = float(min(h.history.get("val_loss", [1e9])))

        rows.append({
            "tag": cfg["tag"],
            "head_units": cfg["head_units"],
            "dropout": cfg["dropout"],
            "lr": cfg["lr"],
            "best_val_acc": best_val_acc,
            "best_val_loss": best_val_loss,
            "seconds": round(time.time() - t0, 2),
        })

        print(f"{cfg['tag']:<14} -> best val_acc={best_val_acc:.4f} | best val_loss={best_val_loss:.4f}")

    tune_df = pd.DataFrame(rows).sort_values("best_val_acc", ascending=False).reset_index(drop=True)
    display(tune_df)

    best = tune_df.iloc[0].to_dict()
    EFFNET_HEAD_UNITS = int(best["head_units"])
    EFFNET_DROPOUT   = float(best["dropout"])
    EFFNET_LR_FROZEN = float(best["lr"])

    print("\nChosen hyperparameters for full EfficientNet training:")
    print("EFFNET_HEAD_UNITS =", EFFNET_HEAD_UNITS)
    print("EFFNET_DROPOUT    =", EFFNET_DROPOUT)
    print("EFFNET_LR_FROZEN  =", EFFNET_LR_FROZEN)
    print("EFFNET_LR_FT      =", EFFNET_LR_FT, "(fine-tuning lr; change if you want)")
else:
    print("Tuning skipped. Using defaults:")
    print("EFFNET_HEAD_UNITS =", EFFNET_HEAD_UNITS)
    print("EFFNET_DROPOUT    =", EFFNET_DROPOUT)
    print("EFFNET_LR_FROZEN  =", EFFNET_LR_FROZEN)
    print("EFFNET_LR_FT      =", EFFNET_LR_FT)


In [ ]:
# ===== Cell 19: EfficientNetB0 ---- FROZEN ----  =====

# Use tuned hyperparameters if defined, otherwise fall back to defaults
EFFNET_HEAD_UNITS = globals().get('EFFNET_HEAD_UNITS', 0)
EFFNET_DROPOUT    = globals().get('EFFNET_DROPOUT', 0.25)
EFFNET_LR_FROZEN  = globals().get('EFFNET_LR_FROZEN', 1e-3)
EFFNET_LR_FT      = globals().get('EFFNET_LR_FT', 1e-4)

base = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE, 3),
)

base.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = data_augment(inputs)

x = base(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
if EFFNET_HEAD_UNITS and EFFNET_HEAD_UNITS > 0:
    x = layers.Dense(int(EFFNET_HEAD_UNITS), activation="relu")(x)

x = layers.Dropout(float(EFFNET_DROPOUT))(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

effnet = keras.Model(inputs, outputs, name="EffNetB0")

effnet.compile(
    optimizer=keras.optimizers.Adam(float(EFFNET_LR_FROZEN)),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

effnet.summary()
print("Base trainable:", base.trainable)

In [ ]:
# ===== Cell 20: Visualize model architectures =====


import sys, subprocess, shutil

def _pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])

# Ensure pydot
try:
    import pydot  # noqa: F401
except Exception:
    _pip_install("pydot")

# Ensure graphviz 'dot' binary (Colab supports apt-get)
if shutil.which("dot") is None:
    try:
        subprocess.check_call(["apt-get", "-qq", "update"])
        subprocess.check_call(["apt-get", "-qq", "install", "graphviz"])
    except Exception as e:
        print("Graphviz install failed (plot_model may not work):", e)

from tensorflow.keras.utils import plot_model
from IPython.display import Image, display

# ScratchCNN
plot_model(
    scratch,
    to_file="ScratchCNN_arch.png",
    show_shapes=True,
    show_layer_names=True,
    dpi=120
)
display(Image("ScratchCNN_arch.png"))

# EfficientNetB0
plot_model(
    effnet,
    to_file="EffNetB0_arch.png",
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=120
)
display(Image("EffNetB0_arch.png"))


In [ ]:
# ===== Cell 21: Train EfficientNetB0 ---- FROZEN ---- =====
EPOCHS_FROZEN = 8

cb2 = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6),
]

hist_eff_frozen = effnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    class_weight=class_weights,
    callbacks=cb2,
    verbose=1
)

plot_history(hist_eff_frozen, "EffNetB0 (Frozen)")

In [ ]:
# ===== Cell 22: Fine-tune top layers ===== ---- UNFROZEN ----
base.trainable = True

# Keep BatchNorm frozen (reduces instability when fine-tuning)
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

effnet.compile(
    optimizer=keras.optimizers.Adam(float(EFFNET_LR_FT)),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS_FT = 8

cb3 = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-7),
]

hist_eff_ft = effnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    class_weight=class_weights,
    callbacks=cb3,
    verbose=1
)

plot_history(hist_eff_ft, "EffNetB0 (Fine-tuned)")

In [ ]:
# ===== Cell 23: Evaluate EfficientNetB0  =====
metrics_eff = eval_model(effnet, test_ds, y_test_idx, title="EffNetB0")
print("EffNet metrics:", metrics_eff)

In [ ]:
# ===== Cell 24: Critical-analysis notes from TEST results =====

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import display, Markdown

# ── Shared style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor":   "#1a1a1a",
    "axes.edgecolor":   "#333333",
    "axes.labelcolor":  "#cccccc",
    "xtick.color":      "#888888",
    "ytick.color":      "#888888",
    "text.color":       "#eeeeee",
    "grid.color":       "#2a2a2a",
    "grid.linestyle":   "--",
    "axes.titlesize":   13,
    "axes.titleweight": "bold",
    "font.family":      "monospace",
})
CNN_COLOR  = "#a78bfa"
EFF_COLOR  = "#38bdf8"
BAD_COLOR  = "#f87171"
GOOD_COLOR = "#34d399"


# ── 0) Narrative ──────────────────────────────────────────────────────────────
scratch_acc = metrics_scratch['acc']
scratch_f1  = metrics_scratch['f1_macro']
eff_acc     = metrics_eff['acc']
eff_f1      = metrics_eff['f1_macro']
gap         = (eff_acc - scratch_acc) * 100

narr = f"""
## Critical Analysis

The results show a clear performance gap between the two approaches.
ScratchCNN achieved a test accuracy of **{scratch_acc:.4f}** (macro F1: **{scratch_f1:.4f}**),
while EfficientNetB0 reached **{eff_acc:.4f}** accuracy (macro F1: **{eff_f1:.4f}**).
This ~**{gap:.1f} pp** gap demonstrates the strong advantage of transfer learning for 38-class image classification.

EfficientNetB0 was pretrained on ImageNet (1.2M images), giving it rich hierarchical visual features.
ScratchCNN must learn all representations from only 43,444 images — insufficient for a deep network
across 38 classes. Even with BatchNormalization and a two-layer Dense head (512→256), the scratch
model is capacity-limited compared to the pretrained backbone.

ScratchCNN showed moderate overfitting — train accuracy kept climbing while val accuracy fluctuated.
EarlyStopping mitigated this by restoring the best weights. EfficientNetB0 showed minimal overfitting
throughout, benefiting from pretrained regularization and a conservative fine-tuning LR of 1e-4.

The hardest classes for ScratchCNN were visually similar Tomato diseases (Spider mites, Target Spot,
Early blight, Late blight) which share overlapping symptom appearances. EfficientNetB0 largely resolves
these confusions. The dataset is heavily imbalanced (Orange___Haunglongbing: 4,405 vs Potato___healthy: 69),
so balanced class weights were applied during training.

**Further improvements could include:**
(a) Testing on real-field images beyond controlled PlantVillage conditions,
(b) Grad-CAM visualisations to interpret which leaf regions drive predictions,
(c) Exploring stronger backbones (EfficientNetB3/V2) for the hardest classes.
"""
display(Markdown(narr))


# ── 1) Metrics Summary Table ──────────────────────────────────────────────────
display(Markdown("---\n## 📊 Results Table"))
df_metrics = pd.DataFrame([
    {"Model": "ScratchCNN",     **metrics_scratch},
    {"Model": "EfficientNetB0", **metrics_eff},
])[["Model","acc","f1_macro","f1_weighted","auc_macro_ovr"]]
df_metrics.columns = ["Model","Accuracy","F1 Macro","F1 Weighted","AUC (OVR)"]
display(df_metrics.round(4))


# ── 2) Learning Curves ────────────────────────────────────────────────────────
display(Markdown("---\n## 📈 Learning Curves"))

def plot_learning_curves(hist_s, hist_e):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle("Learning Curves — ScratchCNN vs EfficientNetB0",
                 fontsize=14, fontweight="bold", color="#eeeeee")
    configs = [
        (axes[0,0], hist_s, "accuracy", "val_accuracy", "ScratchCNN — Accuracy",     CNN_COLOR),
        (axes[0,1], hist_e, "accuracy", "val_accuracy", "EfficientNetB0 — Accuracy", EFF_COLOR),
        (axes[1,0], hist_s, "loss",     "val_loss",     "ScratchCNN — Loss",         CNN_COLOR),
        (axes[1,1], hist_e, "loss",     "val_loss",     "EfficientNetB0 — Loss",     EFF_COLOR),
    ]
    for ax, hist, train_key, val_key, title, color in configs:
        h = getattr(hist, "history", {})
        epochs = range(1, len(h.get(train_key, [])) + 1)
        if train_key in h:
            ax.plot(epochs, h[train_key], color=color, linewidth=2, label="Train", alpha=0.9)
        if val_key in h:
            ax.plot(epochs, h[val_key], color="#ffffff", linewidth=2,
                    label="Val", linestyle="--", alpha=0.8)
        ax.set_title(title, color="#eeeeee")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(train_key.capitalize())
        ax.legend(fontsize=9)
        ax.grid(True)
        # Annotate gap at last epoch
        if train_key == "accuracy" and train_key in h and val_key in h:
            t_last = h[train_key][-1]
            v_last = h[val_key][-1]
            gap_pp = (t_last - v_last) * 100
            ax.annotate(f"Gap: {gap_pp:.1f}pp",
                        xy=(len(epochs), t_last),
                        xytext=(-60, -20), textcoords="offset points",
                        fontsize=9,
                        color=BAD_COLOR if gap_pp > 5 else GOOD_COLOR,
                        arrowprops=dict(arrowstyle="->", color="#888888"))
    plt.tight_layout()
    plt.show()

# Use fine-tune history for EffNet (most informative phase)
_eff_hist_for_curves = None
for _n in ["hist_eff_ft", "hist_eff", "hist_ft", "hist_finetune"]:
    if _n in globals():
        _eff_hist_for_curves = globals()[_n]
        break

if "hist_scratch" in globals() and _eff_hist_for_curves is not None:
    plot_learning_curves(hist_scratch, _eff_hist_for_curves)
else:
    display(Markdown("⚠️ `hist_scratch` or EffNet history not found — run training cells first."))


# ── 3) Overfitting Summary ────────────────────────────────────────────────────
display(Markdown("---\n## ⚡ Overfitting Quick Check"))

def _gap(hist, key="accuracy"):
    if hist is None: return None
    h = getattr(hist, "history", {})
    tr = h.get(key, [])
    va = h.get("val_" + key, [])
    if not tr or not va: return None
    return float(tr[-1] - va[-1])

rows_of = []

# ScratchCNN
if "hist_scratch" in globals():
    g = _gap(hist_scratch)
    if g is not None:
        h = hist_scratch.history
        rows_of.append({
            "Model":            "ScratchCNN",
            "Phase":            "Full Training",
            "Train Acc (last)": round(h["accuracy"][-1], 4),
            "Val Acc (last)":   round(h["val_accuracy"][-1], 4),
            "Gap (pp)":         round(g * 100, 2),
            "Verdict":          "⚠️ Overfitting" if g > 0.05 else "✅ Healthy"
        })

# EffNet frozen phase
if "hist_eff_frozen" in globals():
    g = _gap(hist_eff_frozen)
    if g is not None:
        h = hist_eff_frozen.history
        rows_of.append({
            "Model":            "EfficientNetB0",
            "Phase":            "Frozen",
            "Train Acc (last)": round(h["accuracy"][-1], 4),
            "Val Acc (last)":   round(h["val_accuracy"][-1], 4),
            "Gap (pp)":         round(g * 100, 2),
            "Verdict":          "⚠️ Overfitting" if g > 0.05 else "✅ Healthy"
        })

# EffNet fine-tune phase
if "hist_eff_ft" in globals():
    g = _gap(hist_eff_ft)
    if g is not None:
        h = hist_eff_ft.history
        rows_of.append({
            "Model":            "EfficientNetB0",
            "Phase":            "Fine-tune",
            "Train Acc (last)": round(h["accuracy"][-1], 4),
            "Val Acc (last)":   round(h["val_accuracy"][-1], 4),
            "Gap (pp)":         round(g * 100, 2),
            "Verdict":          "⚠️ Overfitting" if g > 0.05 else "✅ Healthy"
        })

if rows_of:
    display(pd.DataFrame(rows_of))
else:
    hist_vars = [k for k in globals() if "hist" in k.lower()]
    display(Markdown(f"⚠️ No history found. Variables with 'hist': `{hist_vars}`"))


# ── 4) Get predictions (reuse if already computed) ───────────────────────────
def get_preds(model, ds):
    y_true, y_pred = [], []
    for xb, yb in ds:
        y_true.append(np.argmax(yb.numpy(), axis=1))
        y_pred.append(np.argmax(model.predict(xb, verbose=0), axis=1))
    return np.concatenate(y_true), np.concatenate(y_pred)

yt_s = y_true_s if "y_true_s" in globals() else None
yp_s = y_pred_s if "y_pred_s" in globals() else None
yt_e = y_true_e if "y_true_e" in globals() else None
yp_e = y_pred_e if "y_pred_e" in globals() else None

# Fall back to y_test_idx if available (from Cell 25 style)
if yt_s is None and "y_test_idx" in globals() and "scratch" in globals():
    probs = scratch.predict(test_ds, verbose=0)
    yp_s  = probs.argmax(axis=1)
    yt_s  = y_test_idx

if yt_e is None and "y_test_idx" in globals() and "effnet" in globals():
    probs = effnet.predict(test_ds, verbose=0)
    yp_e  = probs.argmax(axis=1)
    yt_e  = y_test_idx


# ── 5) Worst 10 Classes — side by side ───────────────────────────────────────
display(Markdown("---\n## 🔴 Worst 10 Classes by F1-Score"))

def get_worst(yt, yp, n=10):
    rep = classification_report(yt, yp, output_dict=True, zero_division=0)
    df  = pd.DataFrame(rep).T
    df  = df[~df.index.isin(["accuracy","macro avg","weighted avg"])]
    df.index = [x.split("___")[-1][:28] for x in df.index]
    return df.sort_values("f1-score").head(n)[["precision","recall","f1-score","support"]]

if yt_s is not None and yt_e is not None:
    worst_s = get_worst(yt_s, yp_s)
    worst_e = get_worst(yt_e, yp_e)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle("Worst 10 Classes by F1-Score", fontsize=14, fontweight="bold", color="#eeeeee")

    for ax, worst, title, color in [
        (axes[0], worst_s, "ScratchCNN",     CNN_COLOR),
        (axes[1], worst_e, "EfficientNetB0", EFF_COLOR),
    ]:
        bars = ax.barh(worst.index[::-1], worst["f1-score"][::-1],
                       color=color, alpha=0.85, edgecolor="#222222")
        for bar, val in zip(bars, worst["f1-score"][::-1]):
            if val < 0.80:
                bar.set_color(BAD_COLOR)
                bar.set_alpha(0.9)
        ax.set_xlim(0, 1.05)
        ax.set_title(title, color="#eeeeee")
        ax.set_xlabel("F1-Score")
        ax.axvline(x=0.8, color="#555555", linestyle="--", linewidth=1, label="0.8 threshold")
        ax.legend(fontsize=8)
        ax.grid(True, axis="x")
        for bar, val in zip(bars, worst["f1-score"][::-1]):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f"{val:.3f}", va="center", fontsize=8, color="#eeeeee")
    plt.tight_layout()
    plt.show()

    display(Markdown("### ScratchCNN — Worst 10 Classes"))
    display(worst_s.round(4))
    display(Markdown("### EfficientNetB0 — Worst 10 Classes"))
    display(worst_e.round(4))
else:
    display(Markdown("⚠️ Could not compute worst classes. Ensure `scratch`/`effnet` and `test_ds` exist."))


# ── 6) Confusion Matrix — Top 15 Most Confused Classes ───────────────────────
display(Markdown("---\n## 🔵 Confusion Matrix (Top 15 Most Confused Classes)"))

def plot_top_confused_cm(yt, yp, title, top_n=15):
    cm = confusion_matrix(yt, yp)
    cm_nodiag = cm.copy()
    np.fill_diagonal(cm_nodiag, 0)
    confused_idx = np.sort(np.argsort(cm_nodiag.sum(axis=1))[-top_n:])

    cm_sub = cm[np.ix_(confused_idx, confused_idx)]
    labels_sub = [class_names[i].split("___")[-1][:22] for i in confused_idx]

    row_sums = cm_sub.sum(axis=1, keepdims=True)
    cm_norm  = np.where(row_sums > 0, cm_sub / row_sums, 0)

    fig, ax = plt.subplots(figsize=(12, 10))
    fig.patch.set_facecolor("#0f0f0f")
    ax.set_facecolor("#1a1a1a")

    im = ax.imshow(cm_norm, cmap="YlOrRd", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(top_n)); ax.set_yticks(range(top_n))
    ax.set_xticklabels(labels_sub, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(labels_sub, fontsize=8)
    ax.set_xlabel("Predicted Label", labelpad=10)
    ax.set_ylabel("True Label", labelpad=10)
    ax.set_title(f"{title} — Top {top_n} Most Confused Classes (Normalized)", pad=15)

    for i in range(top_n):
        for j in range(top_n):
            val = cm_norm[i, j]
            if val > 0.01:
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=7, color="black" if val > 0.5 else "white")
    plt.tight_layout()
    plt.show()

if yt_s is not None:
    plot_top_confused_cm(yt_s, yp_s, "ScratchCNN")
if yt_e is not None:
    plot_top_confused_cm(yt_e, yp_e, "EfficientNetB0")

In [ ]:
# ===== Cell 25: Save models + class_names.json (zip for download) =====
from google.colab import files
import shutil

OUT_DIR = Path("/content/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

scratch.save(OUT_DIR / "CNN.keras")
effnet.save(OUT_DIR / "EffNetB0.keras")

with open(OUT_DIR / "class_names.json", "w", encoding="utf-8") as f:
    json.dump({"class_names": class_names}, f, indent=2)

zip_path = Path("/content/outputs_models.zip")
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(zip_path).replace(".zip",""), "zip", OUT_DIR)
print("Saved ZIP:", zip_path)

files.download(str(zip_path))

In [ ]:
# ===== Cell 26: Quick single-image inference helper =====
def load_for_infer(path: str):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return tf.expand_dims(img, 0)

def predict_one(model, path: str):
    x = load_for_infer(path)
    probs = model.predict(x, verbose=0)[0]
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx])

# Example:
# sample_path = test_df.iloc[0]["filepath"]
# print(sample_path, "=>", predict_one(effnet, sample_path))

## Submission checklist

**Dataset source URL (required by the brief):**  
Paste the dataset URL you used here (and make sure the lecturer can access it):  
- Dataset URL: **<https://www.kaggle.com/datasets/mohitsingh1804/plantvillage>**  

**Export this notebook to HTML (required by the brief):**  
Run the next cell to generate an `.html` file, then download it from the file panel.

In [ ]:
# Cell 27: Export notebook to HTML =====
# In Colab: after this runs, a .html file appears in the left Files panel.
# You can right-click it and Download.

NOTEBOOK_IN = "Planet_Disease_Classification.ipynb"

# If you renamed the notebook, update NOTEBOOK_IN above.
OUT_BASE = NOTEBOOK_IN.replace(".ipynb", "")
!jupyter nbconvert --to html "{NOTEBOOK_IN}" --output "{OUT_BASE}"

print("Created:", OUT_BASE + ".html")
